# All GPU runs in one session (issues #10, #12, #13)
Order matters: the paraphrase files must exist before the zero-shot and RoBERTa `paraphrase*` settings can be scored.

1. paraphrase attack (#13)  2. zero-shot ppl / logrank / fastdetectgpt (#12)  3. RoBERTa (#10)  4. metrics  5. one zip to download.

**Settings:** GPU T4, Internet on, dataset `team20-m4-processed` attached. Use **Save Version -> Save & Run All (Commit)** so it keeps running with the browser closed; then download the zip from the version's Output tab. No logic in this notebook: every cell calls a script in `src/`.

**What this step does:** downloads the project code from GitHub, installs the Python packages, links the attached dataset into `data/processed`, and checks that every id in the split files exists in the data. It is needed because a fresh Kaggle session starts empty.

**What you should see:** the `ls` listing shows `m4_en.parquet`, and the split check finishes without an error.

**How long:** quick, but it depends on the network (clone and pip install need Internet). No measured time is recorded for this step.

In [ ]:
!git clone -b main https://github.com/NIRMAL20062/generated_text_detection.git
%cd generated_text_detection
# If pip breaks torch, comment this line out: Kaggle's preinstalled packages are enough.
!pip install -q -r requirements.txt
# The dataset mounts under /kaggle/input/... (path differs by account), so find it.
!mkdir -p data && ln -sfn "$(dirname "$(find /kaggle/input -name m4_en.parquet | head -1)")" data/processed
!ls -l data/processed/   # must show m4_en.parquet
!python -m src.data --check-splits

## 1. Paraphrase attack (#13)

**What this step does:** runs the paraphrase attack first: it rewrites 500 machine texts and 200 human texts with a T5 paraphraser and writes `outputs/paraphrase/paraphrased.parquet`, the split files and the quality file. It must come first because the zero-shot and RoBERTa `paraphrase*` settings need these files.

**What you should see:** generation progress, then the new files under `outputs/paraphrase/`, `splits/` and `results/`.

**How long:** about 7 minutes on a Kaggle T4 GPU (399.9 s, from `results/paraphrase_quality.json`).

In [ ]:
!python -m src.paraphrase

## 2. Make `data/processed` writable so every script can read `paraphrased.parquet`
`/kaggle/input` is read-only, so link the main dataset file and copy the new parquet next to it. Then re-check that every id in every split file (now including `paraphrase*`) exists.

**Why the order matters:** `/kaggle/input` is read-only, so the scripts could not find `paraphrased.parquet` next to the main data file. This step makes `data/processed` a normal writable folder, links `m4_en.parquet` into it and copies the new parquet in. Run it after the paraphrase step and before any scoring.

**What you should see:** `ls` lists both `m4_en.parquet` and `paraphrased.parquet`, and the split check passes.

**How long:** quick (no measured time; it only links and copies one file).

In [ ]:
!SRC="$(find /kaggle/input -name m4_en.parquet | head -1)"; rm -f data/processed; mkdir -p data/processed; ln -s "$SRC" data/processed/m4_en.parquet; cp outputs/paraphrase/paraphrased.parquet data/processed/paraphrased.parquet
!ls -l data/processed/
!python -m src.data --check-splits

## 3. Zero-shot (#12): scores every setting, including `paraphrase_orig`, `paraphrase`, `paraphrase_human`

**What this step does:** scores every test text with the perplexity detector (`ppl`). A language model reads each text; a low perplexity means the text looks machine-written. It scores all settings (in-distribution, leave-one-generator-out, paraphrase, cross-domain) in one go and writes `results/ppl/<setting>/preds.csv`.

**What you should see:** progress output, then one result folder per setting under `results/ppl`.

**How long:** about 11 minutes on a Kaggle T4 GPU (640.6 s for the whole run, from `results/ppl/indist/run_info.json`).

In [ ]:
!python -m src.zeroshot --method ppl

**What this step does:** same as the perplexity step, but the score is the average log-rank of each word (how highly the language model ranked the word that was actually written). Machine text uses more top-ranked words. Writes `results/logrank/<setting>/preds.csv`.

**What you should see:** progress output, then one result folder per setting under `results/logrank`.

**How long:** about 11 minutes on a Kaggle T4 GPU (645.1 s for the whole run, from `results/logrank/indist/run_info.json`).

In [ ]:
!python -m src.zeroshot --method logrank

**What this step does:** runs Fast-DetectGPT, the fallback for DetectGPT. It compares how likely the text is under the model with how likely typical alternative texts would be. By default it scores only the DetectGPT subset and the paraphrase test ids (add `--full` to score everything), which is why it is much shorter.

**What you should see:** progress output, then folders under `results/fastdetectgpt`.

**How long:** about 1 minute on a Kaggle T4 GPU (50.5 s, from `results/fastdetectgpt/indist/run_info.json`).

In [ ]:
!python -m src.zeroshot --method fastdetectgpt

### Checkpoint zip (download this first if the session is cut short)

**What this step does:** zips all zero-shot results and the paraphrase files into `checkpoint_zeroshot_paraphrase.zip`. It is a safety copy: if the session is cut during the long RoBERTa part, you can still download this.

**What you should see:** a zip file listed in `/kaggle/working`.

**How long:** quick (no measured time).

In [ ]:
!zip -rq /kaggle/working/checkpoint_zeroshot_paraphrase.zip results/ppl results/logrank results/fastdetectgpt results/paraphrase_quality.json splits/paraphrase.json splits/paraphrase_human.json outputs/paraphrase/paraphrased.parquet
!ls -lh /kaggle/working/*.zip

## 4. RoBERTa (#10), in the order of the issue. `indist` first (its model is reused), then LOGO, paraphrase, cross-domain.

**What this step does:** trains RoBERTa on the in-distribution split and saves the model for reuse by the next steps.

**What you should see:** training progress, then `results/roberta/indist/`.

**How long:** about 9 minutes on a Kaggle T4 GPU (462.4 s training, 516.5 s whole run, from `results/roberta/indist/run_info.json`).

In [ ]:
!python -m src.train_roberta --setting indist

**What this step does:** does not train. It scores the 3 paraphrase settings (`paraphrase_orig`, `paraphrase`, `paraphrase_human`) with the saved `indist` model. These files exist because the paraphrase step ran first.

**What you should see:** 3 result folders under `results/roberta/`.

**How long:** 3 runs of about 20 seconds each on a Kaggle T4 GPU (20.6 to 21.3 s, from `results/roberta/paraphrase*/run_info.json`).

In [ ]:
!python -m src.train_roberta --setting paraphrase_orig --predict-only --model outputs/roberta/indist/model
!python -m src.train_roberta --setting paraphrase --predict-only --model outputs/roberta/indist/model
!python -m src.train_roberta --setting paraphrase_human --predict-only --model outputs/roberta/indist/model

**What this step does:** leave-one-generator-out. For each of 5 generators (chatGPT, davinci, cohere, dolly, bloomz) it trains on the other four and tests on the held-out one. The model is not saved.

**What you should see:** 5 training runs, each writing `results/roberta/logo_<generator>/`.

**How long:** 5 runs of about 6 minutes each on a Kaggle T4 GPU (348 to 354 s per run, from `results/roberta/logo_*/run_info.json`), so about 30 minutes in total.

In [ ]:
for gen in ["chatGPT", "davinci", "cohere", "dolly", "bloomz"]:
    !python -m src.train_roberta --setting logo_{gen} --no-save

**What this step does:** cross-domain test. For each of 5 domains (wikipedia, wikihow, reddit, arxiv, peerread) it trains on the other domains and tests on the held-out one. The model is not saved.

**What you should see:** 5 training runs, each writing `results/roberta/xdomain_<domain>/`.

**How long:** 5 runs of about 8 minutes each on a Kaggle T4 GPU (469 to 476 s per run, from `results/roberta/xdomain_*/run_info.json`), so about 40 minutes in total.

In [ ]:
for dom in ["wikipedia", "wikihow", "reddit", "arxiv", "peerread"]:
    !python -m src.train_roberta --setting xdomain_{dom} --no-save

## 5. Metrics with the shared evaluator (`src/eval.py` is on `main`)

**What this step does:** computes the metrics for every `preds.csv` of the four methods (ppl, logrank, fastdetectgpt, roberta) with `src/eval.py` and prints one summary line each.

**What you should see:** one summary line per setting, and no line starting with `FAILED`.

**How long:** quick (no measured time; no GPU needed).

In [ ]:
import glob, subprocess, sys
for method in ("ppl", "logrank", "fastdetectgpt", "roberta"):
    for p in sorted(glob.glob(f"results/{method}/*/preds.csv")):
        r = subprocess.run([sys.executable, "-m", "src.eval", p], capture_output=True, text=True)
        print((r.stdout.strip().splitlines() or [""])[0] if r.returncode == 0 else f"FAILED {p}: {r.stderr.strip().splitlines()[-1]}")

## 6. Final zip: everything to commit, in one file

**What this step does:** zips everything to commit (all four methods' results, the paraphrase quality file, split files and the paraphrased parquet) into `all_results.zip`, then lists it.

**What you should see:** `all_results.zip` in `/kaggle/working`; download it from the version's Output tab.

**How long:** quick (no measured time).

In [ ]:
!zip -rq /kaggle/working/all_results.zip results/ppl results/logrank results/fastdetectgpt results/roberta results/paraphrase_quality.json splits/paraphrase.json splits/paraphrase_human.json outputs/paraphrase/paraphrased.parquet
!ls -lh /kaggle/working/*.zip